# Mutual Fund Portfolio Optimization

I intend to analyze and optimize my mutual fund portfolio using this personal project. By fetching the historical NAV data for the mutual funds in my portfolio, returns and risks are estimated and mean-variance optimization is carried out using two methods - PyPortfolioOpt and from scratch using scipy. The results are the compared to check how reliable the portfolio optimizer is.

## Phase 1: Data Fetch and Cleaning

Data is fetched from the AMFI scheme master, funds are identified using ISINs and the respective NAV history is downloaded.

In [ ]:
# Importing relevant libraries
import requests
import pandas as pd
import time
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from scipy.optimize import minimize
from pypfopt import EfficientFrontier, expected_returns, risk_models

In [ ]:
# Defining custom color palette
navy = LinearSegmentedColormap.from_list(
    "navy",
    ['#EEF1F5', '#C3CEDE', '#8EA0BD', '#4E6792','#273C5E']
)

In [ ]:
# Getting the complete list of AMFI schemes (scheme code and scheme name)
resp = requests.get('https://api.mfapi.in/mf')
resp.raise_for_status() # Check to find the status of the request

schemes = pd.DataFrame(resp.json())
print(f"Total schemes fetched: {len(schemes)}")

In [ ]:
# Checking the downloaded data
schemes.head(15)

In [ ]:
# Load portfolio holdings from the data file
portfolio = pd.read_excel('data/Mutual Fund Portfolio.xlsx')
print(portfolio.shape)
portfolio.head(17)

In [ ]:
print(portfolio.columns.to_list())

In [ ]:
# Building the Mutual Fund and ISIN mapping 
my_funds = dict(zip(portfolio['MUTUAL FUND'], portfolio['ISIN']))
my_isins = list(my_funds.values())

print(f'{len(my_funds)} mutual funds loaded from the data file')

In [ ]:
# Matching the mutual funds in the data file against the AMFI master using ISIN as key
matched = schemes[schemes['isinGrowth'].isin(my_isins)].copy()
print(f'Matched {len(matched)} out of {len(my_funds)} ISINs')  

In [ ]:
matched[['schemeCode','schemeName','isinGrowth']]

In [ ]:
# Mapping fund name to AMFI scheme code using ISIN as join key
isin_to_name = {isin:name for name, isin in my_funds.items()}
matched['fund_name'] = matched['isinGrowth'].map(isin_to_name)

fund_codes = dict(zip(matched['fund_name'], matched['schemeCode']))
fund_codes

In [ ]:
# Defining a function to fetch the NAV history of the shortlisted functions, one fund at a time
def fetch_nav(scheme_code):
    url = f'https://api.mfapi.in/mf/{scheme_code}'
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()["data"]
    
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'], format = "%d-%m-%Y")
    df["nav"] = df["nav"].astype(float)
    df = df.set_index("date").sort_index()

    return df[['nav']]

In [ ]:
# Testing the function on one sample Mutual Fund
test_code = list(fund_codes.values())[0]
test = fetch_nav(test_code)
print(f"Scheme {test_code}: {len(test)} rows, {test.index.min().date()} to {test.index.max().date()}")
test.tail()

In [ ]:
# Fetching the NAV history for all 17 funds
nav_data = {}

for name, code in fund_codes.items():
    nav_data[name] = fetch_nav(code)['nav'].rename(name)
    print(f"{name}: {len(nav_data[name])} rows, {nav_data[name].index.min().date()} to {nav_data[name].index.max().date()}")
    time.sleep(0.3)

print(f"\n Fetched NAV history for {len(nav_data)} mutual funds")

In [ ]:
# Combine all NAV series into one single dataframe, aligned on the union of dates
nav_matrix = pd.concat(nav_data.values(), axis = 1)
print('Full matrix shape: ', nav_matrix.shape)

# Using a common window so that only those which has data for all the 17 mutual funds are retained
nav_common = nav_matrix.dropna()
print("Common window shape: ",nav_common.shape)

print("Common window: ", nav_common.index.min().date(),  " to ", nav_common.index.max().date())

In [ ]:
# Option B: Dropping HSBC ELSS Tax Saver Fund to extend the common history
nav_matrix_noHSBC = nav_matrix.drop(columns = ['HSBC ELSS Tax Saver Fund Direct Growth'])
nav_common_B = nav_matrix_noHSBC.dropna()
print("Option B Window shape: ",nav_common_B.shape)

print("Option B Window: ", nav_common_B.index.min().date(), " to ", nav_common_B.index.max().date())

In [ ]:
# compute daily simple returns for both windows
returns_A = nav_common.pct_change().dropna()
returns_B = nav_common_B.pct_change().dropna()

print('Returns A Shape: ', returns_A.shape)
print('Returns B Shape: ', returns_B.shape)

returns_A.tail()

In [ ]:
# Defining a function to calculate return and risk statistics

TRADING_DAYS = 252

def annualise(returns):
    annual_return = returns.mean() * TRADING_DAYS
    annual_volatility = returns.std() * (TRADING_DAYS ** 0.5)
    summary = pd.DataFrame({
        "annual_return":annual_return,
        "annual_volatility":annual_volatility
    }
    )
    summary['sharpe_naive'] = summary['annual_return'] / summary['annual_volatility']
    return summary.sort_values("sharpe_naive", ascending = False)
    

In [ ]:
# Calculating the annualised return and volatility for option A
stats_A = annualise(returns_A)
stats_A.style.format("{:0.2%}", subset = ['annual_return','annual_volatility']).format("{:0.2f}", subset = ['sharpe_naive'])

In [ ]:
# Calculating the annualised return and volatility for option B
stats_B = annualise(returns_B)
stats_B.style.format("{:0.2%}", subset = ['annual_return','annual_volatility']).format("{:0.2f}", subset = ['sharpe_naive'])

In [ ]:
# Calculating the covariance and correlation matrices for option A
covariance_A = returns_A.cov() * TRADING_DAYS
correlation_A = returns_A.corr()

print("Covariance Shape: ", covariance_A.shape)
correlation_A.shape


In [ ]:
# Plotting the covariance and correlation matrices

fig, ax = plt.subplots(figsize = (11,9))
im = ax.imshow(correlation_A, cmap = navy, vmin = 0.6, vmax = 1.0, aspect = "auto")

# Renaming the labels for readability
labels = [c.replace(" Fund","").replace(" Direct","").replace(" Growth","").replace(" Plan","").strip() for c in correlation_A.columns]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels (labels, rotation = 90, fontsize = 8)
ax.set_yticklabels (labels, fontsize = 8)

# Annotating each cell of the heatmap with correlation value
for i in range(len(labels)):
    for j in range(len(labels)):
        val = correlation_A.iloc[i,j]
        ax.text(j, i, f"{val:0.2f}", ha = "center", va = "center", fontsize = 6, color = "#2C3A1C" if val < 0.85 else "#F4F6EC")

fig.colorbar(im, ax = ax, shrink = 0.8, label = "Correlation")
ax.set_title("Fund Return Correlations (Window A, approx. 3.75 years)", fontsize = 12, pad = 12)
plt.tight_layout()
fig.savefig('outputs/correlation_heatmap_A.png', dpi = 150, bbox_inches = "tight")
plt.show()


In [ ]:
# Calculating the covariance and correlation matrices for option B
covariance_B = returns_B.cov() * TRADING_DAYS
correlation_B = returns_B.corr()

print("Covariance Shape: ", covariance_B.shape)
correlation_B.shape


In [ ]:
# Plotting the covariance and correlation matrices

fig, ax = plt.subplots(figsize = (11,9))
im = ax.imshow(correlation_A, cmap = navy, vmin = 0.6, vmax = 1.0, aspect = "auto")

# Renaming the labels for readability
labels = [c.replace(" Fund","").replace(" Direct","").replace(" Growth","").replace(" Plan","").strip() for c in correlation_B.columns]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels (labels, rotation = 90, fontsize = 8)
ax.set_yticklabels (labels, fontsize = 8)

# Annotating each cell of the heatmap with correlation value
for i in range(len(labels)):
    for j in range(len(labels)):
        val = correlation_B.iloc[i,j]
        ax.text(j, i, f"{val:0.2f}", ha = "center", va = "center", fontsize = 6, color = "#2C3A1C" if val < 0.85 else "#F4F6EC")

fig.colorbar(im, ax = ax, shrink = 0.8, label = "Correlation")
ax.set_title("Fund Return Correlations (Window B, approx. 5 years)", fontsize = 12, pad = 12)
plt.tight_layout()
fig.savefig('outputs/correlation_heatmap_B.png', dpi = 150, bbox_inches = "tight")
plt.show()

In [ ]:
# Defining the constants
risk_free_rate = 0.07 # India 10Y G-sec ~ 7%
trading_days = 252

In [ ]:
# Defining a function to return annualised returns and annualised volatility
def portfolio_performance (weights, mean_returns, cov_matrix):
    ann_return = np.sum(mean_returns * weights) * trading_days
    ann_vol = np.sqrt(weights.T @ cov_matrix @ weights) * np.sqrt(trading_days)
    return ann_return, ann_vol

In [ ]:
# Defining a function to calculate the negative Sharpe ratio
def neg_sharpe (weights, mean_returns, cov_matrix, rf = risk_free_rate):
    r, v = portfolio_performance(weights, mean_returns, cov_matrix)
    return -(r - rf) / v

In [ ]:
# Defining a function to calculate the portfolio volatility
def portfolio_volatility(weights, mean_returns, cov_matrix):
    return portfolio_performance(weights, mean_returns, cov_matrix)[1]

In [ ]:
# Defining the function to optimize the portfolio
def optimize_portfolio(returns, objective = 'sharpe'):
    mean_returns = returns.mean()
    cov_matrix = returns.cov()
    n = len(returns.columns)

    # Constraints: sum of all weights should be 1. 
    constraints = {'type':"eq", "fun":lambda w: np.sum(w)-1}

    # Bounds: Each of the weights should be between 0 and 1
    bounds = tuple((0,1) for _ in range(n))

    # Starting guess: Assuming equal weights for all funds
    w0 = np.array([1/n] * n)

    if objective == "sharpe":
        obj = neg_sharpe
    else:
        obj = portfolio_volatility

    result = minimize(obj, w0,
                      args = (mean_returns, cov_matrix),
                      method = "SLSQP",
                      bounds = bounds,
                      constraints = constraints)

    weights = pd.Series(result.x, index = returns.columns)
    r, v = portfolio_performance(result.x, mean_returns, cov_matrix)
    sharpe = (r - risk_free_rate) / v
    return weights, r, v, sharpe

# Max-Sharpe portfolio for Window A
w_sharpe_A, ret_A, vol_A, sh_A = optimize_portfolio(returns_A, "sharpe")
print(f"Max-Sharpe (Window A): return = {ret_A:0.2%}, volatility = {vol_A:0.2%}, Sharpe = {sh_A:0.2f}")
print(f"Weights sum to : {w_sharpe_A.sum():0.4f}")
print("\nNon-zero holdings: ")
w_sharpe_A[w_sharpe_A>0.001].sort_values(ascending=False)


In [ ]:
# Minimum Variance portfolio for Window A
w_minvol_A, ret_mv_A, vol_mv_A, sh_mv_A = optimize_portfolio(returns_A,"min_vol")
print(f"Min-Variance(Window A): return = {ret_mv_A:0.2%}, volatility = {vol_mv_A:0.2%}, Sharpe = {sh_mv_A:0.2f}")
print(f"Weights sum to : {w_minvol_A.sum():0.4f}")
print("\nNon-zero holdings: ")
w_minvol_A[w_minvol_A>0.001].sort_values(ascending=False)

In [ ]:
# Max-Sharpe portfolio for Window B
w_sharpe_B, ret_B, vol_B, sh_B = optimize_portfolio(returns_B, "sharpe")
print(f"Max-Sharpe (Window B): return = {ret_B:0.2%}, volatility = {vol_B:0.2%}, Sharpe = {sh_B:0.2f}")
print(f"Weights sum to : {w_sharpe_B.sum():0.4f}")
print("\nNon-zero holdings: ")
w_sharpe_B[w_sharpe_B>0.001].sort_values(ascending=False)


In [ ]:
# Minimum Variance portfolio for Window B
w_minvol_B, ret_mv_B, vol_mv_B, sh_mv_B = optimize_portfolio(returns_B,"min_vol")
print(f"Min-Variance(Window B): return = {ret_mv_B:0.2%}, volatility = {vol_mv_B:0.2%}, Sharpe = {sh_mv_B:0.2f}")
print(f"Weights sum to : {w_minvol_B.sum():0.4f}")
print("\nNon-zero holdings: ")
w_minvol_B[w_minvol_B>0.001].sort_values(ascending=False)

In [ ]:
# validating the optimization results with PyPortfolioOpt
mu = expected_returns.mean_historical_return(nav_common, frequency= trading_days)
S = risk_models.sample_cov(nav_common, frequency= trading_days)

ef = EfficientFrontier(mu, S, weight_bounds=(0,1))
ef.max_sharpe(risk_free_rate=risk_free_rate)
weights_ppo = ef.clean_weights()

print("PyPortfolioOpt Max-Sharpe (Window A): ")
for fund, w in sorted(weights_ppo.items(), key = lambda x: -x[1]):
    if w > 0.001:
        print(f"{fund}: {w:0.4f}")

ret_ppo, vol_ppo, sharpe_ppo = ef.portfolio_performance(risk_free_rate=risk_free_rate)
print(f"Performance: return = {ret_ppo:0.2%}, volatility = {vol_ppo:0.2%}, Sharpe = {sharpe_ppo:0.2f}")


In [ ]:
# Current portfolio weights
current_weights = pd.Series(
    portfolio['WEIGHTS'].values,
    index = portfolio["MUTUAL FUND"].values
)

current_weights = current_weights.reindex(returns_A.columns)
print("Weights sum to: ", round(current_weights.sum(),4))
print("Any missing (NaN)?:", current_weights.isna().any())
current_weights.sort_values(ascending = False)

In [ ]:
# Evaluating current portfolio performance
mean_returns_A = returns_A.mean()
cov_A = returns_A.cov()
cur_ret, cur_vol = portfolio_performance(current_weights.values, mean_returns_A, cov_A)
cur_sharpe = (cur_ret - risk_free_rate) / cur_vol
print(f"Current Portfolio: returns: {cur_ret:0.2%}, vol = {cur_vol:0.2%}, Sharpe: {cur_sharpe:0.2f}")
print(f"Max Sharpe Optimal: returns: {ret_A:0.2%}, vol = {vol_A:0.2%}, Sharpe: {sh_A:0.2f}")
print(f"Minimum Variance: returns: {ret_mv_A:0.2%}, vol = {vol_mv_A:0.2%}, Sharpe: {sh_mv_A:0.2f}")


In [ ]:
# Plotting the Efficient Frontier
np.random.seed(42)
n_portfolios = 5000
n_assets = len(returns_A.columns)

results = np.zeros((3, n_portfolios))
for i in range(n_portfolios):
    w = np.random.random(n_assets)
    w /= w.sum()
    r, v = portfolio_performance(w, mean_returns_A, cov_A)
    results[0,i] = r
    results[1,i] = v
    results[2,i] = (r - risk_free_rate) / v

fig, ax = plt.subplots(figsize = (12,7))

sc = ax.scatter(results[1], results[0], c = results[2], cmap = navy, s = 8, alpha = 0.5)
fig.colorbar(sc, label = "Sharpe Ratio")

ax.scatter(cur_vol, cur_ret, marker = 'X', s = 260, c = '#C0392B', edgecolors='black', linewidth = 1, label = f"Current (Sharpe: {cur_sharpe:0.2f})", zorder=5)

ax.scatter(vol_A, ret_A, marker = '*', s = 420, c = '#F1C40F', edgecolors='black', linewidth = 1, label = f"Max-Sharpe (Sharpe: {sh_A:0.2f})", zorder=5)

ax.scatter(vol_mv_A, ret_mv_A, marker = 'D', s = 160, c = '#2C3A1C', edgecolors='black', linewidth = 1, label = f"Min-Variance (Sharpe: {sh_mv_A:0.2f})", zorder=5)

ax.set_xlabel("Annualized Volatility (Risk)")
ax.set_ylabel("Annualized Return")
ax.set_title("Efficient Frontier, My Portfolio vs Optimal (Window A, ~3.75 years)", fontsize = 13)
ax.legend(loc = 'lower right', framealpha = 0.9)
plt.tight_layout()
plt.savefig('outputs/efficient_frontier.png', dpi = 150, bbox_inches = "tight")
plt.show()

In [ ]:
# Comparing the current and optimal weights
comparison = pd.DataFrame({
    "Current":current_weights,
    "Max-Sharpe":w_sharpe_A
}).fillna(0)

comparison.index = [c.replace(" Fund","").replace(" Direct","").replace(" Growth","").replace(" Plan","").strip() for c in comparison.index]
comparison = comparison.sort_values("Current",ascending=True)

fig, ax = plt.subplots(figsize = (12,8))
y = np.arange(len(comparison))
h = 0.4
ax.barh(y - h/2, comparison['Current'], height=h, label = 'Current', color = '#C0392B')
ax.barh(y + h/2, comparison['Max-Sharpe'], height = h, label = 'Max-Sharpe optimal', color = '#4E6792')
ax.set_yticks(y)
ax.set_yticklabels(comparison.index, fontsize = 8)
ax.set_xlabel("Portfolio Weight")
ax.set_title('Current vs Optimal Allocation (Window A)', fontsize = 13)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/weights_comparison.png',dpi = 150, bbox_inches = 'tight')
plt.show()

The above optimizer produced clean, specific weights. Now, let me stress test whether the weights are reliable. To begin with, I am doing a bootstrapped resampling. This will answer the extent to which the output weights are dependent on the exact sample of days observed.

In [ ]:
# Bootstrapped resampling daily returns with replacement, with oprimizer running each time. The distribution of max-Sharpe weights per fund is saved as a dataframe for analyis and visualization

n_boot = 1000
n_obs = len(returns_A)

boot_weights = []

rng = np.random.default_rng(42)

for b in range(n_boot):
    # Sample row indices with replacement
    idx = rng.integers(0, n_obs, size = n_obs)
    sample = returns_A.iloc[idx]

    try:
        w, _, _, _ = optimize_portfolio(sample, "sharpe")
        boot_weights.append(w)

    except Exception:
        continue

boot_df = pd.DataFrame(boot_weights)
print(f"Completed {len(boot_df)} successful bootstrap runs")
summary = boot_df.describe().T[['mean','std','min','max']].sort_values("mean",ascending=False)
summary.style.format("{:0.1%}")

In [ ]:
# Visualizing the distribution of each fund's bootstrapped optimal weights using box plots

mean_w = boot_df.mean().sort_values(ascending=False)
top_funds = mean_w[mean_w > 0.005].index

plot_data = boot_df[top_funds]
short_labels = [c.replace(" Fund","").replace(" Direct","").replace(" Growth","").replace(" Plan","").strip() for c in plot_data.columns]

fig, ax = plt.subplots(figsize = (12,6))

bp = ax.boxplot([plot_data[c] for c in plot_data.columns],
                vert = False,
                patch_artist=True,
                showfliers=True,
                flierprops= dict(marker = 'o', markersize = 2, alpha = 0.3))

for patch in bp['boxes']:
    patch.set_facecolor("#4E6792")
    patch.set_alpha(0.7)

ax.set_yticklabels(short_labels, fontsize = 9)
ax.set_xlabel("Bootstrapped optimal weights (1000 resamples)")
ax.set_title("The 'optimal' weight is a distribution, not a number", fontsize = 13)
ax.axvline(0, color = 'grey', linewidth = 0.5)
plt.tight_layout()
plt.savefig('outputs/bootstrapped_weights.png', dpi = 150, bbox_inches = "tight")
plt.show()

In [ ]:
# Covariance shrinkage 

def optimize_shrunk(returns_sample):
    mu = returns_sample.mean() * TRADING_DAYS
    S = risk_models.CovarianceShrinkage(returns_sample, frequency=TRADING_DAYS, returns_data=True).ledoit_wolf()
    ef = EfficientFrontier(mu, S, weight_bounds=(0,1))
    ef.max_sharpe(risk_free_rate=risk_free_rate)
    return pd.Series(ef.clean_weights())

# Bootstrapping with shrinkage

boot_weights_shrunk = []
for b in range(n_boot):
    idx = rng.integers(0, len(returns_A), size = len(returns_A))
    sample = returns_A.iloc[idx]
    try:
        boot_weights_shrunk.append(optimize_shrunk(sample))
    except Exception:
        continue

book_shrunk_df = pd.DataFrame(boot_weights_shrunk).fillna(0)
print(f"Completed {len(book_shrunk_df)} shrinkage bootstrap runs")

# Compare standard deviation of weights (raw vs. shrunk)

comparison_std = pd.DataFrame({
    "std_raw" : boot_df.std(),
    "std_shrunk" : book_shrunk_df.std()
}
).sort_values("std_raw",ascending = False)

comparison_std.style.format("{:0.1%}")

In [ ]:
# Optimization using the Sortino ratio
# Function to calculate the annualized downside deviation, returns the count of below target funds
def downside_deviation(returns, weights, target = 0.0):
    port_returns = returns @ weights
    downside = np.minimum(port_returns-target,0)
    dd = np.sqrt(np.mean(downside**2)) * np.sqrt(TRADING_DAYS)
    return dd

# Function to calcuate the negative Sortino ratio. Minimizing this maximises the positive Sortino ratio

def neg_sortino(weights, returns, rf = risk_free_rate):
    ann_return = np.sum(returns.mean() * weights) * TRADING_DAYS
    dd = downside_deviation(returns, weights)
    if dd == 0:
        return 0
    return -(ann_return - rf) / dd

# Function to optimize the portfolio using Sortino's ratio

def optimize_sortino(returns):
    n = len(returns.columns)
    constraints = {'type':'eq',"fun":lambda w: np.sum(w)-1}
    bounds = tuple((0,1) for _ in range(n))
    w0 = np.array([1/n] * n)
    result = minimize(neg_sortino, w0, args = (returns,),
                      method = 'SLSQP', bounds = bounds, constraints = constraints)
    weights = pd.Series(result.x, index = returns.columns)
    return weights

# Sortino-optimal portfolio for Window A
w_sortino_A = optimize_sortino(returns_A)
print("Sortino-optimal portfolio holdings for Window A:")
print(w_sortino_A[w_sortino_A > 0.001].sort_values(ascending=False))

In [ ]:
# Comparing active vs. index funds. Objective here is to add index funds to the portfolio list and re-run the optimizer. If any of the index funds grab weights, it is a clear indicator that the investment is not optimal.

for term in ['UTI Nifty 50 Index', 'Motilal Oswal Nifty Midcap 150 Index', 'Nippon India Nifty Smallcap 250 Index']:
    hits = schemes[schemes['schemeName'].str.contains(term, case = False, na = False)
                   & schemes['schemeName'].str.contains('Direct', case = False, na = False)
                   & schemes['schemeName'].str.contains('Growth', case = False, na = False)]

    print(hits[['schemeCode','schemeName']].to_string(index=False))

In [ ]:
# fetching the NAV for the three index funds and building the extended portfolio set for analysis

index_codes = {
    "UTI Nifty 50 Index": 120716,
    "Motilal Oswal Nifty Midcap 150 Index": 147622,
    "Nippon India Nifty Smallcap 250 Index": 148519
}

index_nav = {}

for name, code in index_codes.items():
    index_nav[name] = fetch_nav(code)['nav'].rename(name)
    print(f"{name}: {len(index_nav[name])} rows, ", f"{index_nav[name].index.min().date()} to {index_nav[name].index.max().date()}")
    time.sleep(0.3)

In [ ]:
# Combining the 17 active funds and the 3 index funds to rerun the optimizer
all_nav = {**nav_data, **index_nav}
nav_all = pd.concat(all_nav.values(), axis = 1)
nav_all_common = nav_all.dropna()
returns_all = nav_all_common.pct_change().dropna()

print('Combined common window: ', returns_all.index.min().date()," to ", returns_all.index.max().date())
print("Shape: ",returns_all.shape)

# Max-sharpe optimization for the combined set
w_with_index, return_with_index, vol_with_index, sharpe_with_index = optimize_portfolio(returns_all, "sharpe")
print(f"Max-Sharpe with Index funds: return = {return_with_index:0.2%}, volatility = {vol_with_index:0.2%}, Sharpe = {sharpe_with_index:0.2f}")
print("Holdings with non-zero weights: ")
print(w_with_index[w_with_index > 0.0001].sort_values(ascending=False)) 

## Why I would not act naively on the optimization results?

When I ran the first optimizer, the results pointed to a direction which would reduce my portfolio of 17 funds to that of 3 (Parag Parikh, Quant Small Cap and Motilal Oswal). An achievable Sharpe ratio of 0.88 seemed too big to ignore compared to the current Sharpe ratio of 0.50. I was tempted to reshuffle my portfolio solely based on these results, but I held back after much deliberation, because of four major reasons:

1. When I stress tested the reliability of the 'optimal' weights using bootstrapped resampling, I realized that the weights were not numbers but rather a wide distribution. The resampling was done using 1000 samples and Parag Parikh's weight ranged from 0 to 100% during this process. For every significant fund (weight greater than 0), the standard deviation of the weights roughly equalled their respective means. 

2. When I ran the optimizer for two time periods - 3.75 years and 5 years, the results were highly inconsistent. Parag Parikh's weight dropped from 44% to 0% while Motilal Oswal's spiked from 16% to 70%. This sensitivity to the time window was alarming.

3. I also ran a check with replacing Sharpe with Sortino as the risk measure. The optimizer returned the same three funds, but with different weights. This also proves that the definition of the 'optimal' portfolio depends on my risk appetite as well.

4. To ensure the method is fool-proof, I also tried Ledoit-Wolf shrinkage fix to reduce the instability. However, the results were largely unchanged. While shrinkage fixed the covariance noise, the instability mostly arises from the noisy return estimates which are not addressed by the shrinkage fix.


## My Takeaways

1. The bootstrapped resampling clearly excluded all the three Axis equity funds by assigning a zero weight in all the samples. I should relook at my investments in these funds, and at least stop or reduce the SIPs if not closing and reinvesting elsewhere.

2. The correlation matrix indicated very high correlation (0.9+) among small-cap, tech, large and midcap funds. This indicates a structural redundancy in the portfolio and I believe this is the end result of a series of investments not based on logic, data or even gut feelings, but hearsay and recommendations.

3. The active-versus-index funds comparison was a relief. The optimizer returned my active funds alone, an otherwise result would have made me question my entire investment strategy. However, this is by design as the comparison spoke only for the three funds picked by the optimizer and not my entire portfolio.

The above analysis can be considered as the very first step in the portfolio optimization process, giving directional view of the performance of the funds.

## Next Steps

1. The immediate next step is to review the structural redundancy and consistent underperformers. While there are several external factors that also need consideration such as ELSS lock-in, capital-gains tax, investment horizon, short term and long term goals, I am more inclined to do away with the underperformers and limit my investment to just one tech fund.

2. I also intend to keep a close-watch on the funds that are given zero weights in the above optimization results. I want to do this review regularly, maybe on a quarterly basis to improve the efficiency of my investments.

Optimizers can always come up with precise, confident outputs. The real challenge is to know when to trust them blindly and when not to. Domain knowledge and experience play a key role in ensuring how the results are interpreted.